In [ ]:
import os
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
chest_xray/
    train/
        NORMAL/
        PNEUMONIA/
    test/
        NORMAL/
        PNEUMONIA/
    val/
        NORMAL/
        PNEUMONIA/

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
DATA_DIR = "./chest_xray"

train_dataset = ImageFolder(
    root=os.path.join(DATA_DIR, "train"),
    transform=transform
)

test_dataset = ImageFolder(
    root=os.path.join(DATA_DIR, "test"),
    transform=transform
)

print("Классы:", train_dataset.classes)

print("Обучающих изображений:", len(train_dataset))
print("Тестовых изображений:", len(test_dataset))

In [ ]:
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_subset))
print("Validation:", len(val_subset))
print("Test:", len(test_dataset))

In [ ]:
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Замораживаем все слои
for param in model.parameters():
    param.requires_grad = False

# Заменяем классификатор
model.fc = nn.Linear(512, 2)

# Переносим модель на устройство
model = model.to(device)

print(model.fc)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.0001
)

num_epochs = 5

train_losses = []
val_losses = []

In [ ]:
best_val_loss = float('inf')
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_subset)

    train_losses.append(epoch_train_loss)

    # Проверка на валидационной выборке
    model.eval()

    val_running_loss = 0.0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

    epoch_val_loss = val_running_loss / len(val_subset)

    val_losses.append(epoch_val_loss)

    # Сохраняем лучшие веса
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        best_model_weights = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Эпоха [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

# Загружаем лучшие веса
model.load_state_dict(best_model_weights)

print("\nОбучение завершено!")

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, num_epochs + 1),
    train_losses,
    marker='o',
    label='Training Loss'
)

plt.plot(
    range(1, num_epochs + 1),
    val_losses,
    marker='s',
    label='Validation Loss'
)

plt.title("ResNet18 Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.xticks(range(1, num_epochs + 1))
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())


accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

print("Результаты тестирования:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['NORMAL', 'PNEUMONIA'],
    yticklabels=['NORMAL', 'PNEUMONIA']
)

plt.xlabel("Предсказанный класс")
plt.ylabel("Реальный класс")

plt.title("Confusion Matrix - ResNet18")

plt.show()

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': train_dataset.classes,
    'model_name': 'ResNet18'
}, 'pneumonia_resnet18.pth')

print("Модель сохранена: pneumonia_resnet18.pth")

checkpoint = torch.load(
    'pneumonia_resnet18.pth',
    map_location=device,
    weights_only=True
)

loaded_model = models.resnet18(weights=None)

loaded_model.fc = nn.Linear(512, 2)

loaded_model.load_state_dict(
    checkpoint['model_state_dict']
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

print("Модель успешно загружена!")
print("Классы:", checkpoint['classes'])

In [ ]:

random.seed(42)

random_indices = random.sample(
    range(len(test_dataset)),
    8
)

class_names = test_dataset.classes

# Параметры обратной нормализации
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

model.eval()

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for ax, idx in zip(axes.flat, random_indices):

    image, true_label = test_dataset[idx]

    # Предсказание модели
    with torch.no_grad():

        input_image = image.unsqueeze(0).to(device)

        output = model(input_image)

        predicted_label = torch.argmax(
            output,
            dim=1
        ).item()

    # Обратная нормализация
    image_display = image * std + mean

    image_display = image_display.clamp(0, 1)

    image_display = image_display.permute(1, 2, 0).numpy()

    # Отображение изображения
    ax.imshow(image_display)

    predicted_name = class_names[predicted_label]
    true_name = class_names[true_label]

    correct = predicted_label == true_label

    color = 'green' if correct else 'red'

    ax.set_title(
        f"Predicted: {predicted_name}\n"
        f"Actual: {true_name}",
        color=color,
        fontsize=10
    )

    ax.axis('off')

plt.suptitle(
    "MediScan AI - Pneumonia Predictions",
    fontsize=16
)

plt.tight_layout()

# Сохранение графика
plt.savefig(
    "pneumonia_predictions.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("График сохранён: pneumonia_predictions.png")

In [ ]:
results = pd.DataFrame({
    'Метрика': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score'
    ],
    'Значение': [
        round(accuracy, 4),
        round(precision, 4),
        round(recall, 4),
        round(f1, 4)
    ]
})

display(results)